In [31]:
! pip install -q langchain langgraph langchain-mistralai scikit-learn numpy pandas


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import json
import math
import uuid
from datetime import datetime, timezone
from typing import TypedDict, List, Dict, Any, Optional

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

from langchain_mistralai import ChatMistralAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage

from langgraph.graph import StateGraph, END

In [2]:
from datasets import load_dataset

print('Chargement du corpus ViDoRe V3 Pharmaceuticals...')
# cast_columns=False + keep_in_memory=False : les images ne sont PAS décodées en RAM
# Elles restent en bytes bruts et sont converties à la volée lors de l'encodage
corpus_ds  = load_dataset('vidore/vidore_v3_pharmaceuticals', 'corpus',  split='test')
queries_ds = load_dataset('vidore/vidore_v3_pharmaceuticals', 'queries', split='test')
qrels_ds   = load_dataset('vidore/vidore_v3_pharmaceuticals', 'qrels',   split='test')

Chargement du corpus ViDoRe V3 Pharmaceuticals...


In [3]:
# Convert datasets to dict for fast lookup

corpus_lookup = {
    str(x["corpus_id"]): x
    for x in corpus_ds
}

query_lookup = {
    str(x["query_id"]): x
    for x in queries_ds
}

print("Corpus example:", list(corpus_lookup.items())[:1])
print("Query example:", list(query_lookup.items())[:1])

Corpus example: [('0', {'corpus_id': 0, 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=2000x1125 at 0x1A039EA66E0>, 'doc_id': 'final_ce_dental_caries', 'markdown': 'U.S. FOOD& DRUG ADMINISTRATION\nFDA\nFDA Drug Topics:\nAn Update on Transmucosal\nBuprenorphine and Dental Caries\nMark A. Liberatore, PharmD, RAC\nCommander, United States Public Health Service\nDeputy Director for Safety\nDivision of Anesthe Addiction Medicine, and Pain Medicine\nOffice of New Drugs, Center for Drug Evaluation and Research,\nU.S. Food and Drug Administration', 'page_number_in_doc': 0})]
Query example: [('0', {'query_id': 0, 'query': 'In which year were the Kefauver-Harris Amendments passed to make randomized controlled trials the standard for evaluating clinical efficacy?', 'language': 'english', 'query_types': ['extractive', 'numerical'], 'query_format': 'question', 'content_type': ['Image', 'Text'], 'raw_answers': ['In 1962, the Kefauver-Harris Amendments were passed to make randomized co

In [4]:
os.environ["MISTRAL_API_KEY"] = 'LUNRDu37JwpajYLk97nDLkeLRyJFj1qG'

In [5]:
llm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0
)

In [6]:
TEXT_CORPUS_PATH = "embeddings_textual_corpus.json"
TEXT_QUERY_PATH = "embeddings_textual_queries.json"
VISUAL_CORPUS_PATH = "embeddings_visual_corpus.json"
VISUAL_QUERY_PATH = "embeddings_visual_queries.json"

In [7]:
def load_embedding_dict(path: str) -> Dict[str, np.ndarray]:
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    
    out = {}
    for k, v in raw.items():
        out[str(k)] = np.array(v, dtype=np.float32)
    return out

In [8]:
text_corpus = load_embedding_dict(TEXT_CORPUS_PATH)
text_queries = load_embedding_dict(TEXT_QUERY_PATH)
visual_corpus = load_embedding_dict(VISUAL_CORPUS_PATH)
visual_queries = load_embedding_dict(VISUAL_QUERY_PATH)

print("text corpus:", len(text_corpus))
print("text queries:", len(text_queries))
print("visual corpus:", len(visual_corpus))
print("visual queries:", len(visual_queries))

text corpus: 2313
text queries: 2184
visual corpus: 2313
visual queries: 2184


In [9]:
print("Sample text corpus keys:", list(text_corpus.keys())[:5])
print("Sample text query keys:", list(text_queries.keys())[:5])
print("Sample visual corpus keys:", list(visual_corpus.keys())[:5])
print("Sample visual query keys:", list(visual_queries.keys())[:5])

first_tc = next(iter(text_corpus))
first_tq = next(iter(text_queries))
first_vc = next(iter(visual_corpus))
first_vq = next(iter(visual_queries))

print("Text corpus dim:", text_corpus[first_tc].shape)
print("Text query dim:", text_queries[first_tq].shape)
print("Visual corpus dim:", visual_corpus[first_vc].shape)
print("Visual query dim:", visual_queries[first_vq].shape)

Sample text corpus keys: ['0', '1', '2', '3', '4']
Sample text query keys: ['0', '1', '2', '3', '4']
Sample visual corpus keys: ['0', '1', '2', '3', '4']
Sample visual query keys: ['0', '1', '2', '3', '4']
Text corpus dim: (768,)
Text query dim: (768,)
Visual corpus dim: (512,)
Visual query dim: (512,)


In [10]:
def normalize(v: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(v)
    if n < 1e-12:
        return v
    return v / n

In [11]:
common_corpus_ids = sorted(set(text_corpus.keys()) & set(visual_corpus.keys()))
common_query_ids = sorted(set(text_queries.keys()) & set(visual_queries.keys()))

print("Common corpus ids:", len(common_corpus_ids))
print("Common query ids:", len(common_query_ids))

text_corpus_matrix = np.vstack([normalize(text_corpus[i]) for i in common_corpus_ids])
visual_corpus_matrix = np.vstack([normalize(visual_corpus[i]) for i in common_corpus_ids])

text_query_matrix = np.vstack([normalize(text_queries[i]) for i in common_query_ids])
visual_query_matrix = np.vstack([normalize(visual_queries[i]) for i in common_query_ids])

print("Text corpus matrix:", text_corpus_matrix.shape)
print("Visual corpus matrix:", visual_corpus_matrix.shape)
print("Text query matrix:", text_query_matrix.shape)
print("Visual query matrix:", visual_query_matrix.shape)

Common corpus ids: 2313
Common query ids: 2184
Text corpus matrix: (2313, 768)
Visual corpus matrix: (2313, 512)
Text query matrix: (2184, 768)
Visual query matrix: (2184, 512)


In [12]:
def retrieve_multimodal_by_query_id(
    query_id: str,
    k: int = 5,
    alpha_text: float = 0.5,
    alpha_visual: float = 0.5
) -> List[Dict[str, Any]]:
    
    query_id = str(query_id)

    tq = normalize(text_queries[query_id]).reshape(1, -1)
    vq = normalize(visual_queries[query_id]).reshape(1, -1)

    text_scores = cosine_similarity(tq, text_corpus_matrix)[0]
    visual_scores = cosine_similarity(vq, visual_corpus_matrix)[0]
    fused_scores = alpha_text * text_scores + alpha_visual * visual_scores

    top_idx = np.argsort(fused_scores)[::-1][:k]

    results = []
    for rank, idx in enumerate(top_idx, start=1):
        corpus_id = common_corpus_ids[idx]
        corpus_item = corpus_lookup.get(corpus_id, {})

        results.append({
            "rank": rank,
            "corpus_id": corpus_id,
            "doc_id": corpus_item.get("doc_id"),
            "page_number": corpus_item.get("page_number"),
            "score_fused": float(fused_scores[idx]),
            "score_text": float(text_scores[idx]),
            "score_visual": float(visual_scores[idx]),
            "markdown": corpus_item.get("markdown", "")[:2000]  # truncate
        })

    return results

In [13]:
def retrieve_text_only_by_query_id(query_id: str, k: int = 5) -> List[Dict[str, Any]]:
    query_id = str(query_id)
    tq = normalize(text_queries[query_id]).reshape(1, -1)
    scores = cosine_similarity(tq, text_corpus_matrix)[0]
    top_idx = np.argsort(scores)[::-1][:k]
    
    return [
        {
            "rank": rank,
            "corpus_id": common_corpus_ids[idx],
            "score": float(scores[idx]),
        }
        for rank, idx in enumerate(top_idx, start=1)
    ]

def retrieve_visual_only_by_query_id(query_id: str, k: int = 5) -> List[Dict[str, Any]]:
    query_id = str(query_id)
    vq = normalize(visual_queries[query_id]).reshape(1, -1)
    scores = cosine_similarity(vq, visual_corpus_matrix)[0]
    top_idx = np.argsort(scores)[::-1][:k]
    
    return [
        {
            "rank": rank,
            "corpus_id": common_corpus_ids[idx],
            "score": float(scores[idx]),
        }
        for rank, idx in enumerate(top_idx, start=1)
    ]

In [14]:
test_query_id = "0"

mm_results = retrieve_multimodal_by_query_id(test_query_id, k=5)
text_results = retrieve_text_only_by_query_id(test_query_id, k=5)
visual_results = retrieve_visual_only_by_query_id(test_query_id, k=5)

print("Multimodal:")
for r in mm_results:
    print(r)

print("\nText only:")
for r in text_results:
    print(r)

print("\nVisual only:")
for r in visual_results:
    print(r)

Multimodal:
{'rank': 1, 'corpus_id': '69', 'doc_id': 'The_Public_Health_Role_of_Drug_Regulation_in_the_US', 'page_number': None, 'score_fused': 0.4559080898761749, 'score_text': 0.6340512633323669, 'score_visual': 0.2777649164199829, 'markdown': "The Age Effectiveness: FDA The 1962 Kefauver-Harris Amendments Not obvious why thalidomide (a safety problem) led to a requirement for demonstrating effectiveness, but the 1962 Act made at least 3 important changes: 1. FDA had to give positive approval before a drug could be marketed 2. A meaningful requirement to study drugs under an IND and an explicit requirement for informed consent (one year before the Declaration of Helsinki). Dr. Francis Kelsey received the President's Award for Distinguished Federal Civilian Service 3. The effectiveness requirement 14"}
{'rank': 2, 'corpus_id': '1210', 'doc_id': 'FDLI_FDA_CDER_PCavazzoni', 'page_number': None, 'score_fused': 0.4003984332084656, 'score_text': 0.4828852713108063, 'score_visual': 0.317911

In [15]:
@tool
def multimodal_search(query_id: str, k: int = 5) -> List[Dict[str, Any]]:
    """Retrieve top-k corpus items using fused textual and visual embeddings."""
    return retrieve_multimodal_by_query_id(query_id=query_id, k=k)

@tool
def textual_search(query_id: str, k: int = 5) -> List[Dict[str, Any]]:
    """Retrieve top-k corpus items using textual embeddings only."""
    return retrieve_text_only_by_query_id(query_id=query_id, k=k)

@tool
def visual_search(query_id: str, k: int = 5) -> List[Dict[str, Any]]:
    """Retrieve top-k corpus items using visual embeddings only."""
    return retrieve_visual_only_by_query_id(query_id=query_id, k=k)

In [16]:
def classify_request(user_request: str) -> str:
    q = user_request.lower()
    complex_markers = [
        "compare", "comparer", "synthèse", "summary", "résumé",
        "risk", "facteurs", "associated", "pattern", "tendance",
        "why", "how", "explain", "severity", "sévère"
    ]
    if any(m in q for m in complex_markers):
        return "COMPLEX"
    return "SIMPLE"

In [17]:
class AgentState(TypedDict, total=False):
    user_request: str
    query_id: str
    route: str
    retrieved_docs: List[Dict[str, Any]]
    observations: List[str]
    final_answer: str
    confidence: str
    requires_human_review: bool
    audit_log: List[Dict[str, Any]]

In [18]:
def log_event(state: AgentState, step_type: str, content: Dict[str, Any]) -> AgentState:
    state.setdefault("audit_log", [])
    state["audit_log"].append({
        "event_id": str(uuid.uuid4()),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "step_type": step_type,
        "content": content
    })
    return state

In [19]:
def router_node(state: AgentState) -> AgentState:
    route = classify_request(state["user_request"])
    state["route"] = route
    state = log_event(state, "THOUGHT", {
        "message": f"Request classified as {route}"
    })
    return state

In [20]:
def retrieval_node(state: AgentState) -> AgentState:
    qid = str(state["query_id"])
    k = 5 if state["route"] == "SIMPLE" else 8

    state = log_event(state, "ACT", {
        "tool": "multimodal_search",
        "arguments": {"query_id": qid, "k": k}
    })

    docs = multimodal_search.invoke({"query_id": qid, "k": k})
    state["retrieved_docs"] = docs

    state = log_event(state, "OBSERVE", {
        "n_results": len(docs),
        "top_results": docs[:3]
    })
    return state

In [21]:
def analysis_node(state: AgentState) -> AgentState:
    user_request = state["user_request"]
    docs = state.get("retrieved_docs", [])

    source_text = "\n\n".join([
        f"[doc={d['doc_id']} page={d['page_number']} score={d['score_fused']:.3f}]\n{d['markdown']}"
        for d in docs
    ])

    prompt = f"""
You are an expert pharmacovigilance analyst.

User request:
{user_request}

Retrieved evidence:
{source_text}

Your task:
- Compare evidence across documents
- Identify risk factors, patterns, or trends
- Distinguish clearly:
    • Observed facts
    • Hypotheses / interpretations
- Highlight uncertainty or missing data

Return:
1. Key observations
2. Risk factors (if any)
3. Confidence level: HIGH / MEDIUM / LOW
4. Need human review: YES / NO
"""

    resp = llm.invoke(prompt).content

    state.setdefault("observations", [])
    state["observations"].append(resp)

    state = log_event(state, "THOUGHT", {
        "message": "Synthesized cross-document analysis",
        "analysis": resp
    })

    return state

In [22]:
def final_answer_node(state: AgentState) -> AgentState:
    user_request = state["user_request"]
    docs = state.get("retrieved_docs", [])
    observations = "\n\n".join(state.get("observations", []))

    references = [
        f"{d['doc_id']} (page {d['page_number']})"
        for d in docs
    ]

    prompt = f"""
You are generating a structured answer for an agentic RAG system.

User request:
{user_request}

Analysis:
{observations}

Sources:
{references}

Write a structured answer:

1. Summary of findings
2. Key risk factors / insights
3. Uncertainty and limitations
4. Source references (doc_id + page)

Be precise and conservative.
"""

    final_answer = llm.invoke(prompt).content
    state["final_answer"] = final_answer

    # Conservative pharma rule
    state["confidence"] = "MEDIUM"
    state["requires_human_review"] = True

    state = log_event(state, "FINAL", {
        "confidence": state["confidence"],
        "requires_human_review": True,
        "answer": final_answer
    })

    return state

In [23]:
graph = StateGraph(AgentState)

graph.add_node("router", router_node)
graph.add_node("retrieve", retrieval_node)
graph.add_node("analyze", analysis_node)
graph.add_node("finalize", final_answer_node)

graph.set_entry_point("router")
graph.add_edge("router", "retrieve")
graph.add_edge("retrieve", "analyze")
graph.add_edge("analyze", "finalize")
graph.add_edge("finalize", END)

agent = graph.compile()

In [28]:
query_id = "42"

state = {
    "user_request": query_lookup[query_id]["query"],
    "query_id": query_id,
    "observations": [],
    "audit_log": []
}

result = agent.invoke(state)

In [29]:
print(result["final_answer"])

### **Structured Answer: Impact of the Outdated OTC Monograph System on Drug Innovation and Safety**

---

#### **1. Summary of Findings**
The **U.S. Over-the-Counter (OTC) monograph system**, largely unchanged since 1972, poses **significant barriers to drug innovation and safety** due to its **regulatory rigidity, understaffing, and slow response to emerging risks**. Key findings include:

- **Innovation Stagnation:**
  - The system’s **lack of updates since 1972** discourages new formulations, delivery methods, or indications, as manufacturers face **no clear pathway for modernization** (CDER_Update_Presentation).
  - The **CARES Act (2020)** introduced reforms (e.g., **Over-the-Counter Monograph User Fee Program [OMUFA]**, administrative orders), but their **long-term impact on innovation remains unproven** (FDLI_FDA_CDER_PCavazzoni).

- **Safety Risks:**
  - **Medication errors** (e.g., name confusion, labeling ambiguities) are a **persistent issue**, leading to **product withdraw

In [30]:
for step in result["audit_log"]:
    print("=" * 80)
    print("STEP TYPE:", step["step_type"])
    print("TIMESTAMP:", step["timestamp_utc"])
    print(json.dumps(step["content"], indent=2, ensure_ascii=False))

STEP TYPE: THOUGHT
TIMESTAMP: 2026-03-25T14:13:45.666182+00:00
{
  "message": "Request classified as SIMPLE"
}
STEP TYPE: ACT
TIMESTAMP: 2026-03-25T14:13:45.666182+00:00
{
  "tool": "multimodal_search",
  "arguments": {
    "query_id": "42",
    "k": 5
  }
}
STEP TYPE: OBSERVE
TIMESTAMP: 2026-03-25T14:13:45.677182+00:00
{
  "n_results": 5,
  "top_results": [
    {
      "rank": 1,
      "corpus_id": "738",
      "doc_id": "DDI_Medication_Errors_CE_March_2020_DMEPA_and_ISMP_combined_slides_FINAL",
      "page_number": null,
      "score_fused": 0.4560834467411041,
      "score_text": 0.6202454566955566,
      "score_visual": 0.2919214367866516,
      "markdown": "Proprietary Name Review FDA Conducts misbranding assessment of the proposed proprietary name OPDP* BEST PRACTICES IN DEVELOPING PROPRIETARY NAMES FOR DRUGS *For OTC products, the misbranding review is conducted by the Office of Nonprescription Drugs (ONPD) Provides misbranding and safety concerns with the proposed proprietary n

In [31]:
gt = [
    x["corpus_id"]
    for x in qrels_ds
    if str(x["query_id"]) == query_id
]

retrieved = [r["corpus_id"] for r in result["retrieved_docs"]]

print("Ground truth:", gt[:5])
print("Retrieved:", retrieved[:5])

Ground truth: [1956, 1959]
Retrieved: ['738', '1956', '2080', '1181', '806']
